In [3]:
pip install  imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached imbalanced_learn-0.14.0-py3-none-any.whl.metadata (8.8 kB)
Using cached imbalanced_learn-0.14.0-py3-none-any.whl (239 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
# -------------------------------
# Cell 1: Imports
# -------------------------------
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
import joblib
import re

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# -------------------------------
# Cell 2: Load dataset
# -------------------------------
data = pd.read_csv('../data/fake_job_postings.csv')
data.fillna('', inplace=True)

# -------------------------------
# Cell 3: Text cleaning function
# -------------------------------
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # remove punctuation
    return text

# -------------------------------
# Cell 4: Combine fields & clean
# -------------------------------
data['text'] = (data['title'] + ' ' + data['company_profile'] + ' ' +
                data['description'] + ' ' + data['requirements']).apply(clean_text)
X = data['text']
y = data['fraudulent']

# -------------------------------
# Cell 5: Train-test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# -------------------------------
# Cell 6: TF-IDF
# -------------------------------
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# -------------------------------
# Cell 7: Handle class imbalance with SMOTE
# -------------------------------
smote = SMOTE(random_state=42)
X_train_tfidf_res, y_train_res = smote.fit_resample(X_train_tfidf, y_train)

# -------------------------------
# Cell 8: Train multiple models
# -------------------------------
models = {
    'Logistic Regression': LogisticRegression(max_iter=500),
    'Random Forest': RandomForestClassifier(n_estimators=200),
    'Naive Bayes': MultinomialNB(),
    'Support Vector Machine': SVC(kernel='linear'),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier()
    
}

for name, model in models.items():
    model.fit(X_train_tfidf_res, y_train_res)
    y_pred = model.predict(X_test_tfidf)
    print(f'--- {name} ---')
    print('Accuracy:', accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))
    print('\n')

# -------------------------------
# Cell 9: Save all models & vectorizer
# -------------------------------
for name, model in models.items():
    filename = f'../models/{name.replace(" ", "_")}.pkl'
    joblib.dump(model, filename)
    print(f'Saved model to {filename}')

joblib.dump(vectorizer, '../models/vectorizer.pkl')
print('Saved vectorizer to ../models/vectorizer.pkl')


--- Logistic Regression ---
Accuracy: 0.9745525727069351
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3403
           1       0.69      0.86      0.76       173

    accuracy                           0.97      3576
   macro avg       0.84      0.92      0.88      3576
weighted avg       0.98      0.97      0.98      3576

[[3337   66]
 [  25  148]]


--- Random Forest ---
Accuracy: 0.9829418344519015
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3403
           1       0.98      0.66      0.79       173

    accuracy                           0.98      3576
   macro avg       0.98      0.83      0.89      3576
weighted avg       0.98      0.98      0.98      3576

[[3401    2]
 [  59  114]]


--- Naive Bayes ---
Accuracy: 0.9437919463087249
              precision    recall  f1-score   support

           0       0.99      0.95      0.97      3403
           1       0.4

['../models/vectorizer.pkl']